### Update `openalex.works.work_authors` — Author ID Matching & Minting

Assigns `author_id` to work_authors records where `author_id IS NULL`. Resolves institution IDs from `work_authors.raw_affiliation_strings` (written by `UpdateWorkAuthors`). Does NOT update affiliations.

In [0]:
DECLARE OR REPLACE VARIABLE max_updated_date TIMESTAMP DEFAULT to_timestamp('1900-01-01');
SET VARIABLE max_updated_date = COALESCE((SELECT MAX(updated_at) - INTERVAL 1 SECOND FROM identifier('openalex' || :env_suffix || '.works.work_authors')), to_timestamp('1900-01-01'));
-- SET VARIABLE max_updated_date = to_timestamp('2025-12-20');
SELECT max_updated_date;

-- oxjob #649 rematch-on-authorship-change. Mode & seat cap come from author_rematch_control
-- (single row; empty table = defaults below). mode: 'log_only' (write author_rematch_log, NO
-- nulls, zero effect on matching) | 'off' | 'on' (apply: worklist + null/insert/delete + rematch).
CREATE TABLE IF NOT EXISTS openalex.authors.author_rematch_control (mode STRING, seat_cap INT);

DECLARE OR REPLACE VARIABLE rematch_mode STRING DEFAULT 'log_only';

SET VARIABLE rematch_mode = COALESCE((SELECT ANY_VALUE(mode) FROM openalex.authors.author_rematch_control), 'log_only');

DECLARE OR REPLACE VARIABLE rematch_seat_cap INT DEFAULT 250000;

SET VARIABLE rematch_seat_cap = COALESCE((SELECT ANY_VALUE(seat_cap) FROM openalex.authors.author_rematch_control), 250000);

CREATE TABLE IF NOT EXISTS openalex.authors.author_rematch_worklist (
    work_id BIGINT, change_classes ARRAY<STRING>, eligible BOOLEAN, est_seats INT, admitted_at TIMESTAMP);

CREATE TABLE IF NOT EXISTS openalex.authors.author_rematch_applied (
    run_date DATE, work_id BIGINT, change_classes ARRAY<STRING>, eligible BOOLEAN, est_seats INT, admitted_at TIMESTAMP);

CREATE TABLE IF NOT EXISTS openalex.authors.author_rematch_pins (
    work_id BIGINT, author_sequence BIGINT, author_id BIGINT, reason STRING, pinned_at TIMESTAMP);


### Rematch-on-authorship-change detection — oxjob #649 (LOG ONLY)

Snapshot `openalex.authors.author_rematch_log`, rebuilt each run (`rematch_mode='log_only'`: **no nulls, no worklist, zero effect on matching**). Two classes over ALL work ids, tagged `eligible`:
- `foreign_family`: bound `author_id` is a **different surname family** from the current `works_base` name — the strict 'very wrong / different person' bar (wave-S foreign-family predicate: surname not shared/contained/1-edit/swapped; display-word intersection empty; #608 judged-same overlays; junk/markup + CJK/Cyrillic/Greek/Arabic abstention). Opus precision ~87% (vs 35% for bare names_compatible). `name_diverged` = current name != stored work_authors name (change-driven subset).
- `size_grew`/`size_shrank`: `works_base` author count != `work_authors` seat count.

Holds: `orcid_anchor`, `curated`. Action IS wired (next cell: worklist + null/insert/delete + batch-gate union) but fires ONLY when `rematch_mode='on'` — under `log_only` the worklist rebuilds EMPTY and every write is a no-op. Gate design RESOLVED 2026-07-22: old works re-enter matching via `author_rematch_worklist` ONLY (blanket id/date gate stays); worklisted records match AND mint normally. Comma-form surname-first names are reordered ('Oliveira, C.' -> 'C. Oliveira') before the display-word intersection (closes the main FP class, ~87%->~95% precision).

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.author_rematch_log AS
WITH base AS (
    SELECT id AS work_id, SIZE(authorships) AS base_n, authorships,
           (id > 7000000000 AND created_date >= to_timestamp('2025-12-20')) AS eligible
    FROM openalex.works.openalex_works_base
    WHERE authorships IS NOT NULL AND SIZE(authorships) > 0 AND rematch_mode <> 'off'
),
seats AS (
    SELECT b.work_id, b.eligible, t.pos AS author_sequence,
           TRIM(t.a.raw_author_name) AS base_name, NULLIF(TRIM(t.a.raw_orcid), '') AS raw_orcid
    FROM base b LATERAL VIEW posexplode(b.authorships) t AS pos, a
),
bound AS (
    SELECT s.work_id, s.eligible, s.author_sequence, s.base_name, s.raw_orcid,
           TRIM(wa.raw_author_name) AS stored_name, wa.author_id
    FROM seats s JOIN openalex.works.work_authors wa
      ON s.work_id = wa.work_id AND s.author_sequence = wa.author_sequence
    WHERE wa.author_id IS NOT NULL
),
prof AS (SELECT id, COALESCE(NULLIF(TRIM(display_name), ''), TRIM(full_name)) AS fn, NULLIF(TRIM(orcid), '') AS orcid
         FROM openalex.authors.openalex_authors),
k AS (
    SELECT b.work_id, b.eligible, b.author_sequence, b.base_name, b.stored_name, b.author_id, b.raw_orcid,
           p.fn AS bound_name, p.orcid AS bound_orcid, NOT (b.base_name <=> b.stored_name) AS name_diverged,
           an_r.match_last AS r_last, an_r.match_first AS r_first, an_p.match_last AS p_last, an_p.match_first AS p_first
    FROM bound b JOIN prof p ON b.author_id = p.id
    JOIN openalex.authors.author_names an_r ON b.base_name = an_r.raw_author_name
    JOIN openalex.authors.author_names an_p ON p.fn = an_p.raw_author_name
    WHERE an_r.match_last IS NOT NULL AND an_p.match_last IS NOT NULL
      -- junk / placeholder / markup names excluded
      AND b.base_name NOT RLIKE '(?i)^(n/?a|null|none|unknown|anonymous|et al\.?|grd)\\b'
      AND NOT (b.base_name RLIKE '<[a-zA-Z/][^>]*>' OR b.base_name RLIKE '(?i)(vcard|begin:|;type=)')
      -- script abstention: CJK + Cyrillic/Greek + Arabic (frozen-parser low confidence)
      AND NOT (b.base_name RLIKE '[Ͱ-ϿЀ-ӿ؀-ۿᄀ-ᇿ぀-ヿ㄰-㆏㐀-䶿一-鿿가-힯豈-﫿]'
               OR p.fn RLIKE '[Ͱ-ϿЀ-ӿ؀-ۿᄀ-ᇿ぀-ヿ㄰-㆏㐀-䶿一-鿿가-힯豈-﫿]')
),
ff AS (   -- FOREIGN FAMILY = 'very wrong' (different surname family), not merely incompatible
    SELECT * FROM k
    WHERE r_last <> p_last AND LENGTH(p_last) >= 2 AND LENGTH(r_last) >= 2
      AND INSTR(p_last, r_last) = 0 AND INSTR(r_last, p_last) = 0
      AND r_last <> COALESCE(p_first, '') AND COALESCE(r_first, '') <> p_last
      -- married-name exemption: full given name agrees -> assume surname change, leave alone
      AND NOT (r_first IS NOT NULL AND r_first = p_first AND LENGTH(r_first) >= 3)
      AND SIZE(ARRAY_INTERSECT(
        FILTER(SLICE(SPLIT(TRIM(REGEXP_REPLACE(TRANSLATE(LOWER(REGEXP_REPLACE(REGEXP_REPLACE(base_name,'^([^,]+),(.*)$','$2 $1'),'[.,\\-]+',' ')),'áàäâãåéèëêíìïîóòöôõøúùüûñçýćčšžł','aaaaaaeeeeiiiioooooouuuuncyccszl'),'\\s+',' ')),' '),2,20), t->LENGTH(t)>=3),
        FILTER(SLICE(SPLIT(TRIM(REGEXP_REPLACE(TRANSLATE(LOWER(REGEXP_REPLACE(REGEXP_REPLACE(bound_name,'^([^,]+),(.*)$','$2 $1'),'[.,\\-]+',' ')),'áàäâãåéèëêíìïîóòöôõøúùüûñçýćčšžł','aaaaaaeeeeiiiioooooouuuuncyccszl'),'\\s+',' ')),' '),2,20), t->LENGTH(t)>=3)
      )) = 0
      AND levenshtein(r_last, p_last) > 1
      AND NOT (levenshtein(r_last, p_last) = 2 AND LEFT(COALESCE(r_first,''),1) = LEFT(COALESCE(p_first,''),1))
      AND NOT EXISTS (SELECT 1 FROM openalex.authors.oxjob608_namepair_j_gemini j WHERE j.full_name=bound_name AND j.raw_name=base_name AND j.judge_json:same_person='true')
      AND NOT EXISTS (SELECT 1 FROM openalex.authors.oxjob608_namepair_j_rules jr WHERE jr.full_name=bound_name AND jr.raw_name=base_name AND jr.same_person)
      AND NOT EXISTS (SELECT 1 FROM openalex.authors.oxjob608_namepair_j_opus jo WHERE jo.full_name=bound_name AND jo.raw_name=base_name AND jo.judge_json:same_person='true')
),
wa_ct AS (SELECT work_id, COUNT(*) AS wa_n FROM openalex.works.work_authors GROUP BY work_id),
size_chg AS (
    SELECT b.work_id, b.eligible, b.base_n, COALESCE(w.wa_n, 0) AS wa_n
    FROM base b LEFT JOIN wa_ct w ON b.work_id = w.work_id WHERE b.base_n <> COALESCE(w.wa_n, 0)
)
SELECT current_date() AS run_date, work_id, author_sequence,
       base_name AS raw_author_name, stored_name, author_id AS prev_author_id, bound_name,
       name_diverged, 'foreign_family' AS change_class,
       CAST(NULL AS INT) AS base_n, CAST(NULL AS INT) AS wa_n, eligible, 'dryrun' AS action,
       CASE WHEN raw_orcid IS NOT NULL AND raw_orcid = bound_orcid THEN 'orcid_anchor'
            WHEN EXISTS (SELECT 1 FROM openalex.works.work_author_claim_curations cc
                         WHERE cc.work_id = ff.work_id AND (TRIM(cc.raw_author_name) = ff.base_name OR cc.author_id = ff.author_id))
                 THEN 'curated'
            WHEN EXISTS (SELECT 1 FROM openalex.authors.author_rematch_pins pin
                         WHERE pin.work_id = ff.work_id AND pin.author_sequence = ff.author_sequence AND pin.author_id = ff.author_id)
                 THEN 'pinned' ELSE NULL END AS hold_reason,
       current_timestamp() AS detected_at
FROM ff
UNION ALL
SELECT current_date(), work_id, CAST(NULL AS BIGINT), CAST(NULL AS STRING), CAST(NULL AS STRING),
       CAST(NULL AS BIGINT), CAST(NULL AS STRING), CAST(NULL AS BOOLEAN),
       CASE WHEN base_n > wa_n THEN 'size_grew' ELSE 'size_shrank' END,
       base_n, wa_n, eligible, 'dryrun', CAST(NULL AS STRING), current_timestamp()
FROM size_chg

### Rematch action — oxjob #649 (gated: fires only when `rematch_mode='on'`)

Worklist-scoped gate relax per resolved design (2026-07-22): old works re-enter matching ONLY via `author_rematch_worklist`; admitted records match AND mint normally. Admission is capped by `rematch_seat_cap` (SEATS, not works: est_seats = ff-nulled + grew-inserts + pre-existing in-range nulls), priority foreign_family > size_grew > size_shrank, eligible first. Writes are surgical: DELETE shrank orphan seats (seq >= base_n, curation-guarded), INSERT missing grew seats (author_id NULL), NULL foreign_family seats (unheld, still bound to the logged author). `author_rematch_applied` keeps the append-only history (debounce/metrics). Standing drain at 250K seats/run is ~20 nights.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.author_rematch_worklist AS
WITH cand AS (
    SELECT work_id, ANY_VALUE(eligible) AS eligible,
           COLLECT_SET(change_class) AS change_classes,
           SUM(CASE WHEN change_class = 'foreign_family' AND hold_reason IS NULL THEN 1 ELSE 0 END) AS ff_seats,
           MAX(CASE WHEN change_class = 'size_grew' THEN base_n - wa_n ELSE 0 END) AS grew_seats,
           MAX(CASE WHEN change_class <> 'foreign_family' THEN base_n END) AS size_base_n
    FROM openalex.authors.author_rematch_log
    WHERE rematch_mode = 'on'
      AND NOT EXISTS (SELECT 1 FROM openalex.authors.author_rematch_applied ap
                      WHERE ap.work_id = author_rematch_log.work_id
                        AND ap.run_date >= DATE_SUB(current_date(), 7))
    GROUP BY work_id
    HAVING SUM(CASE WHEN change_class = 'foreign_family' AND hold_reason IS NULL THEN 1 ELSE 0 END) > 0
        OR ARRAY_CONTAINS(COLLECT_SET(change_class), 'size_grew')
        OR ARRAY_CONTAINS(COLLECT_SET(change_class), 'size_shrank')
),
nulls AS (
    SELECT c.work_id, COUNT(*) AS null_seats
    FROM cand c JOIN openalex.works.work_authors wa ON wa.work_id = c.work_id
    WHERE wa.author_id IS NULL AND (c.size_base_n IS NULL OR wa.author_sequence < c.size_base_n)
    GROUP BY c.work_id
),
ranked AS (
    SELECT c.work_id, c.change_classes, c.eligible,
           CAST(c.ff_seats + c.grew_seats + COALESCE(n.null_seats, 0) AS INT) AS est_seats,
           SUM(c.ff_seats + c.grew_seats + COALESCE(n.null_seats, 0)) OVER (
               ORDER BY CASE WHEN ARRAY_CONTAINS(c.change_classes, 'foreign_family') THEN 0
                             WHEN ARRAY_CONTAINS(c.change_classes, 'size_grew') THEN 1 ELSE 2 END,
                        c.eligible DESC, c.work_id
               ROWS UNBOUNDED PRECEDING) AS cum_seats
    FROM cand c LEFT JOIN nulls n ON c.work_id = n.work_id
)
SELECT work_id, change_classes, eligible, est_seats, current_timestamp() AS admitted_at
FROM ranked WHERE cum_seats <= rematch_seat_cap;

INSERT INTO openalex.authors.author_rematch_applied
SELECT current_date(), work_id, change_classes, eligible, est_seats, admitted_at
FROM openalex.authors.author_rematch_worklist;

DELETE FROM openalex.works.work_authors AS wa
WHERE EXISTS (
        SELECT 1 FROM openalex.authors.author_rematch_worklist wl
        JOIN openalex.authors.author_rematch_log l
          ON l.work_id = wl.work_id AND l.change_class = 'size_shrank'
        WHERE wl.work_id = wa.work_id AND wa.author_sequence >= l.base_n)
  AND NOT EXISTS (
        SELECT 1 FROM openalex.works.work_author_claim_curations cc
        WHERE cc.work_id = wa.work_id
          AND (cc.author_id = wa.author_id OR TRIM(cc.raw_author_name) = TRIM(wa.raw_author_name)))
  AND NOT EXISTS (
        SELECT 1 FROM openalex.authors.author_rematch_pins pin
        WHERE pin.work_id = wa.work_id AND pin.author_sequence = wa.author_sequence
          AND pin.author_id = wa.author_id);

INSERT INTO openalex.works.work_authors
    (work_id, author_sequence, author_id, raw_author_name, raw_affiliation_strings, is_corresponding, created_at, updated_at)
SELECT b.id, t.pos, CAST(NULL AS BIGINT), TRIM(t.a.raw_author_name), t.a.raw_affiliation_strings,
       t.a.is_corresponding, current_timestamp(), current_timestamp()
FROM openalex.works.openalex_works_base b
JOIN openalex.authors.author_rematch_worklist wl
  ON b.id = wl.work_id AND ARRAY_CONTAINS(wl.change_classes, 'size_grew')
LATERAL VIEW POSEXPLODE(b.authorships) t AS pos, a
WHERE NOT EXISTS (
    SELECT 1 FROM openalex.works.work_authors wa
    WHERE wa.work_id = b.id AND wa.author_sequence = t.pos);

MERGE INTO openalex.works.work_authors AS wa
USING (
    SELECT l.work_id, l.author_sequence, l.prev_author_id
    FROM openalex.authors.author_rematch_log l
    JOIN openalex.authors.author_rematch_worklist wl ON l.work_id = wl.work_id
    WHERE l.change_class = 'foreign_family' AND l.hold_reason IS NULL
) s
ON wa.work_id = s.work_id AND wa.author_sequence = s.author_sequence AND wa.author_id = s.prev_author_id
WHEN MATCHED THEN UPDATE SET wa.author_id = NULL, wa.updated_at = current_timestamp();

### Step 1: Get updated works that need matching

In [0]:
-- STEP 1: Create Staging Table — only records that need author matching
-- Resolves institution IDs from work_authors.raw_affiliation_strings (written by affiliations step)
CREATE OR REPLACE TABLE openalex.authors.author_matching_batch AS
WITH raw_exploded AS (
    SELECT 
        id AS work_id,
        created_date,
        POSEXPLODE(authorships) AS (author_sequence, authorship)
    FROM identifier('openalex' || :env_suffix || '.works.openalex_works_base')
    WHERE (updated_date > max_updated_date
           OR id IN (SELECT work_id FROM openalex.authors.author_rematch_worklist))
      AND authorships IS NOT NULL 
      AND SIZE(authorships) > 0
),
-- Only keep authorships that need matching:
-- 1. No author_id assigned yet
-- 2. Meets the ID and date cutoffs for new-era matching
needs_matching AS (
    SELECT r.work_id, r.author_sequence, r.authorship,
           r.authorship.raw_author_name AS raw_author_name,
           wa.raw_affiliation_strings
    FROM raw_exploded r
    INNER JOIN identifier('openalex' || :env_suffix || '.works.work_authors') wa
        ON r.work_id = wa.work_id AND r.author_sequence = wa.author_sequence
    WHERE wa.author_id IS NULL
      AND ((r.work_id > 7000000000 AND r.created_date >= to_timestamp('2025-12-20'))
           OR r.work_id IN (SELECT work_id FROM openalex.authors.author_rematch_worklist))
),

-- Resolve institution IDs from work_authors.raw_affiliation_strings
exploded_ras AS (
    SELECT nm.work_id, nm.author_sequence, nm.authorship, nm.raw_author_name,
           t.raw_affiliation_string
    FROM needs_matching nm
    LATERAL VIEW OUTER EXPLODE(nm.raw_affiliation_strings) t AS raw_affiliation_string
),
resolved_direct_ids AS (
    SELECT 
        e.work_id, e.author_sequence, e.authorship, e.raw_author_name,
        CASE 
            WHEN e.raw_affiliation_string IS NULL THEN NULL
            WHEN asl.institution_ids IS NOT NULL AND SIZE(asl.institution_ids) > 0 
                AND NOT (SIZE(asl.institution_ids) = 1 AND asl.institution_ids[0] = -1) 
                THEN asl.institution_ids
            ELSE NULL
        END AS direct_ids
    FROM exploded_ras e
    LEFT JOIN openalex.institutions.raw_affiliation_strings_institutions_mv asl
        ON e.raw_affiliation_string = asl.raw_affiliation_string
),

-- Expand Lineage (FOR MATCHING ONLY)
expanded_for_matching AS (
    SELECT 
        r.work_id,
        r.author_sequence,
        ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(
            FLATTEN(ARRAY(
                FILTER(ARRAY(r.inst_id_scalar), x -> x IS NOT NULL),
                COALESCE(anc.ancestors, ARRAY())
            ))
        ))) as matching_institution_ids
    FROM (
        SELECT work_id, author_sequence, EXPLODE_OUTER(direct_ids) as inst_id_scalar
        FROM resolved_direct_ids
    ) r
    LEFT JOIN (
        SELECT institution_id, lineage_ids as ancestors
        FROM openalex.institutions.institution_ancestors
    ) anc ON CAST(r.inst_id_scalar AS BIGINT) = anc.institution_id
    GROUP BY r.work_id, r.author_sequence
)

SELECT 
    nm.work_id,
    nm.author_sequence,
    COALESCE(efm.matching_institution_ids, ARRAY()) as all_institution_ids,
    nm.authorship as authorship_struct,
    nm.raw_author_name
FROM needs_matching nm
LEFT JOIN expanded_for_matching efm
    ON nm.work_id = efm.work_id AND nm.author_sequence = efm.author_sequence;

### Step 2: Run Matching Algorithm Over Updated Works with ID over 7000000000

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.pending_author_assignments AS
WITH 
-- 1. ENRICH BATCH DATA
-- Add Signals (Topics, Sources) and parsed name columns to batch data
enriched_batch AS (
  SELECT
    b.work_id,
    b.author_sequence,
    b.raw_author_name,
    b.all_institution_ids,
    TRANSFORM(b.all_institution_ids, x -> CONCAT('https://openalex.org/I', CAST(x AS STRING))) AS institution_ids,
    
    pn.parsed_name.first AS pn_first,
    SUBSTRING(pn.parsed_name.first, 1, 1) AS pn_first_initial,
    pn.parsed_name.middle AS pn_middle,
    pn.parsed_name.last AS pn_last,
    
    COALESCE(wtf.topics, ARRAY()) AS topics,
    
    ARRAY_DISTINCT(
      TRANSFORM(
        FILTER(w.locations, x -> x.source.id IS NOT NULL),
        x -> x.source.id
      )
    ) AS work_source_ids,
    b.authorship_struct.raw_orcid AS incoming_orcid
    
  FROM openalex.authors.author_matching_batch b
  LEFT JOIN openalex.authors.author_names pn 
    ON TRIM(b.raw_author_name) = pn.raw_author_name
  LEFT JOIN openalex.works.work_topics wtf 
    ON b.work_id = wtf.work_id
  -- works_base, NOT openalex_works: openalex_works is rebuilt AFTER this task
  -- in end2end, so joining it misses every new work (work_source_ids empty).
  LEFT JOIN openalex.works.openalex_works_base w 
    ON b.work_id = w.id
),

-- 2. PREPARE MATCHING INPUTS
-- Calculate Block Keys and ID arrays
authors_prepared AS (
  SELECT
    work_id,
    author_sequence,
    raw_author_name,
    pn_first,
    pn_first_initial,
    pn_middle,
    pn_last,
    -- Block Key Generation (parsed_name fields are already normalized)
    CASE
      WHEN pn_last IS NULL THEN NULL
      WHEN pn_first_initial IS NULL OR pn_first_initial = '' THEN pn_last
      ELSE CONCAT(pn_first_initial, ' ', pn_last)
    END AS block_key,
    institution_ids,
    -- Extract Topic IDs
    TRANSFORM(topics, t -> t.id) AS topic_ids,
    work_source_ids,
    incoming_orcid
  FROM enriched_batch
  WHERE raw_author_name IS NOT NULL
),

-- 2b. ORCID MATCHING (global — no block constraint)
-- Match incoming ORCID to any profile holding it. If several profiles share
-- the ORCID (splinters), take the most-cited, then most works, then oldest id.
orcid_matches AS (
  SELECT
    ap.work_id,
    ap.author_sequence,
    COUNT(DISTINCT a.author_id) AS orcid_match_count,
    MAX_BY(a.author_id, STRUCT(COALESCE(a.cited_by_count, 0), COALESCE(a.works_count, 0), -a.author_id)) AS orcid_author_id
  FROM (
    -- Guard: publishers sometimes stamp one author's ORCID on every authorship
    -- of a work. An ORCID appearing on >1 authorship of the same work is
    -- untrustworthy — skip the ORCID tier for those rows (name cascade decides).
    SELECT work_id, author_sequence, incoming_orcid,
           COUNT(*) OVER (PARTITION BY work_id, incoming_orcid) AS orcid_uses_in_work
    FROM authors_prepared
    WHERE incoming_orcid IS NOT NULL
  ) ap
  JOIN openalex.authors.authors_for_matching a
    ON a.orcid = ap.incoming_orcid
  WHERE ap.orcid_uses_in_work = 1
  GROUP BY ap.work_id, ap.author_sequence
),

-- 3. CANDIDATE BLOCKING
blocked_candidates AS (
  SELECT 
    e.work_id,
    e.author_sequence,
    e.raw_author_name,
    e.pn_first,
    e.pn_first_initial,
    e.pn_middle,
    e.pn_last,
    e.block_key,
    e.institution_ids,
    e.topic_ids,
    e.work_source_ids,
    alm.author_id,
    alm.display_name AS candidate_display_name,
    alm.first AS cand_first,
    alm.first_initial AS cand_first_initial,
    alm.middle AS cand_middle,
    alm.last AS cand_last,
    alm.institution_ids as candidate_institution_ids,
    alm.topic_ids as candidate_topic_ids,
    alm.source_ids AS candidate_source_ids,
    alm.works_count
  FROM authors_prepared e
  LEFT JOIN openalex.authors.authors_for_matching alm
    ON alm.block_key = e.block_key
    AND e.block_key != ''
),

with_match_signals AS (
  SELECT
    *,
    NAMED_STRUCT(
      'id', author_id,
      'display_name', candidate_display_name
    ) AS candidate_obj,
    
    (size(institution_ids) > 0 AND size(candidate_institution_ids) > 0 
     AND arrays_overlap(candidate_institution_ids, institution_ids)) as has_inst,
    
    (size(topic_ids) > 0 AND size(candidate_topic_ids) > 0 
     AND arrays_overlap(candidate_topic_ids, topic_ids)) as has_topic,

     (SIZE(work_source_ids) > 0 AND SIZE(candidate_source_ids) > 0
     AND ARRAYS_OVERLAP(candidate_source_ids, work_source_ids)) AS has_source
  FROM blocked_candidates
),

with_name_matches AS (
  SELECT
    *,
    -- 1: Exact Full Name (both have full first, full middle, same last)
    (LENGTH(pn_first) > 1 AND LENGTH(pn_middle) > 1 AND LENGTH(cand_first) > 1 AND LENGTH(cand_middle) > 1
     AND pn_first = cand_first
     AND pn_middle = cand_middle
     AND pn_last = cand_last
    ) as pattern_1_exact_full,

    -- 2: Exact First, Middle Initial match (batch has full first + middle initial only)
    (LENGTH(pn_first) > 1 AND (pn_middle IS NULL OR LENGTH(pn_middle) <= 1)
     AND LENGTH(cand_first) > 1
     AND pn_first = cand_first
     AND pn_last = cand_last
     AND (cand_middle IS NULL OR pn_middle IS NULL OR SUBSTRING(pn_middle, 1, 1) = SUBSTRING(cand_middle, 1, 1))
    ) as pattern_2_exact_first_mid_init,

    -- 3: Initials match to Full (batch has first initial + middle, candidate has full)
    (LENGTH(pn_first) = 1 AND pn_middle IS NOT NULL
     AND LENGTH(cand_first) > 1 AND cand_middle IS NOT NULL
     AND pn_first_initial = cand_first_initial
     AND SUBSTRING(pn_middle, 1, 1) = SUBSTRING(cand_middle, 1, 1)
     AND pn_last = cand_last
    ) as pattern_3_init_mid_init,

    -- 4: First Initial, Middle Initial match (both have only initials, no full names)
    (LENGTH(pn_first) = 1 AND LENGTH(cand_first) = 1
     AND pn_middle IS NOT NULL AND cand_middle IS NOT NULL
     AND LENGTH(pn_middle) <= 1 AND LENGTH(cand_middle) <= 1
     AND pn_first_initial = cand_first_initial
     AND SUBSTRING(pn_middle, 1, 1) = SUBSTRING(cand_middle, 1, 1)
     AND pn_last = cand_last
    ) as pattern_4_first_init_mid_init,

    -- 5: Exact First, Exact Last (no middle)
    (LENGTH(pn_first) > 1 AND LENGTH(cand_first) > 1
     AND pn_first = cand_first
     AND pn_last = cand_last
     AND pn_middle IS NULL
    ) as pattern_5_exact_first_last,

    -- 6: First Initial Only to Full (batch has first initial only, candidate has full first)
    (LENGTH(pn_first) = 1 AND pn_middle IS NULL
     AND LENGTH(cand_first) > 1
     AND pn_first_initial = cand_first_initial
     AND pn_last = cand_last
    ) as pattern_6_first_init_to_full,

    -- 7: First Initial Only (both have only first initial, no middle)
    (LENGTH(pn_first) = 1 AND LENGTH(cand_first) = 1
     AND pn_middle IS NULL AND cand_middle IS NULL
     AND pn_first_initial = cand_first_initial
     AND pn_last = cand_last
    ) as pattern_7_first_init_last,

    -- 8: Full Name to Initial (batch has full first, candidate has only initial)
    (LENGTH(pn_first) > 1 AND LENGTH(cand_first) = 1
     AND pn_first_initial = cand_first_initial
     AND pn_last = cand_last
    ) as pattern_8_full_to_init

  FROM with_match_signals
),

with_any_name_match AS (
  SELECT
    *,
    (pattern_1_exact_full OR pattern_2_exact_first_mid_init OR pattern_3_init_mid_init OR 
     pattern_4_first_init_mid_init OR pattern_5_exact_first_last OR pattern_6_first_init_to_full OR 
     pattern_7_first_init_last OR pattern_8_full_to_init) as any_name_match
  FROM with_name_matches
),

aggregated_counts AS (
  SELECT
    work_id,
    author_sequence,
    raw_author_name,
    block_key,
    institution_ids,
    pn_first,
    pn_first_initial,
    pn_middle,
    pn_last,
    work_source_ids,
    
    -- STRATEGY 1: Name Only (Unique)
    count_if(pattern_1_exact_full) AS s1_n1, count_if(pattern_2_exact_first_mid_init) AS s1_n2,
    count_if(pattern_3_init_mid_init) AS s1_n3, count_if(pattern_4_first_init_mid_init) AS s1_n4,
    count_if(pattern_5_exact_first_last) AS s1_n5, count_if(pattern_6_first_init_to_full) AS s1_n6,
    count_if(pattern_7_first_init_last) AS s1_n7, count_if(pattern_8_full_to_init) AS s1_n8,
    
    -- STRATEGY 2: Name + Institution
    count_if(pattern_1_exact_full AND has_inst) AS s2_n1, count_if(pattern_2_exact_first_mid_init AND has_inst) AS s2_n2,
    count_if(pattern_3_init_mid_init AND has_inst) AS s2_n3, count_if(pattern_4_first_init_mid_init AND has_inst) AS s2_n4,
    count_if(pattern_5_exact_first_last AND has_inst) AS s2_n5, count_if(pattern_6_first_init_to_full AND has_inst) AS s2_n6,
    count_if(pattern_7_first_init_last AND has_inst) AS s2_n7, count_if(pattern_8_full_to_init AND has_inst) AS s2_n8,

    -- STRATEGY 6: Name + Inst + Source
    count_if(pattern_1_exact_full AND has_inst AND has_source) AS s6_n1,
    count_if(pattern_2_exact_first_mid_init AND has_inst AND has_source) AS s6_n2,
    count_if(pattern_5_exact_first_last AND has_inst AND has_source) AS s6_n5,
    count_if(pattern_6_first_init_to_full AND has_inst AND has_source) AS s6_n6,
    count_if(pattern_7_first_init_last AND has_inst AND has_source) AS s6_n7,

    -- STRATEGY 4: Name + Inst + Topic
    count_if(pattern_1_exact_full AND has_inst AND has_topic) AS s4_n1,
    count_if(pattern_2_exact_first_mid_init AND has_inst AND has_topic) AS s4_n2,
    count_if(pattern_5_exact_first_last AND has_inst AND has_topic) AS s4_n5,
    count_if(pattern_6_first_init_to_full AND has_inst AND has_topic) AS s4_n6,
    count_if(pattern_7_first_init_last AND has_inst AND has_topic) AS s4_n7,

    -- STRATEGY 5: Name + Source
    count_if(pattern_1_exact_full AND has_source) AS s5_n1,
    count_if(pattern_2_exact_first_mid_init AND has_source) AS s5_n2,
    count_if(pattern_5_exact_first_last AND has_source) AS s5_n5,
    count_if(pattern_6_first_init_to_full AND has_source) AS s5_n6,
    count_if(pattern_7_first_init_last AND has_source) AS s5_n7,
    count_if(pattern_8_full_to_init AND has_source) AS s5_n8,

    -- STRATEGY 3: Name + Topic
    count_if(pattern_1_exact_full AND has_topic) AS s3_n1,
    count_if(pattern_2_exact_first_mid_init AND has_topic) AS s3_n2,
    count_if(pattern_5_exact_first_last AND has_topic) AS s3_n5,
    
    -- CAPTURE MATCHED OBJECTS
    MAX(CASE WHEN pattern_1_exact_full THEN candidate_obj END) AS match_s1_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init THEN candidate_obj END) AS match_s1_n2,
    MAX(CASE WHEN pattern_5_exact_first_last THEN candidate_obj END) AS match_s1_n5,

    MAX(CASE WHEN pattern_1_exact_full AND has_inst THEN candidate_obj END) AS match_s2_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_inst THEN candidate_obj END) AS match_s2_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND has_inst THEN candidate_obj END) AS match_s2_n5,
    MAX(CASE WHEN pattern_6_first_init_to_full AND has_inst THEN candidate_obj END) AS match_s2_n6,
    MAX(CASE WHEN pattern_8_full_to_init AND has_inst THEN candidate_obj END) AS match_s2_n8,

    MAX(CASE WHEN pattern_1_exact_full AND has_inst AND has_source THEN candidate_obj END) AS match_s6_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_inst AND has_source THEN candidate_obj END) AS match_s6_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND has_inst AND has_source THEN candidate_obj END) AS match_s6_n5,
    MAX(CASE WHEN pattern_6_first_init_to_full AND has_inst AND has_source THEN candidate_obj END) AS match_s6_n6,

    MAX(CASE WHEN pattern_1_exact_full AND has_source THEN candidate_obj END) AS match_s5_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_source THEN candidate_obj END) AS match_s5_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND has_source THEN candidate_obj END) AS match_s5_n5,
    MAX(CASE WHEN pattern_6_first_init_to_full AND has_source THEN candidate_obj END) AS match_s5_n6,
    MAX(CASE WHEN pattern_8_full_to_init AND has_source THEN candidate_obj END) AS match_s5_n8,
    
    MAX(CASE WHEN pattern_1_exact_full AND has_inst AND has_topic THEN candidate_obj END) AS match_s4_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_inst AND has_topic THEN candidate_obj END) AS match_s4_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND has_inst AND has_topic THEN candidate_obj END) AS match_s4_n5,
    MAX(CASE WHEN pattern_6_first_init_to_full AND has_inst AND has_topic THEN candidate_obj END) AS match_s4_n6,

    MAX(CASE WHEN pattern_1_exact_full AND has_topic THEN candidate_obj END) AS match_s3_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_topic THEN candidate_obj END) AS match_s3_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND has_topic THEN candidate_obj END) AS match_s3_n5,
    
    COUNT(author_id) AS total_candidates_in_block,
    COUNT_IF(any_name_match) AS total_name_matches

  FROM with_any_name_match
  GROUP BY work_id, author_sequence, raw_author_name, block_key, institution_ids,
           pn_first, pn_first_initial, pn_middle, pn_last, work_source_ids
),

final_decision AS (
SELECT
  ac.work_id,
  ac.author_sequence,
  ac.block_key,
  raw_author_name,
  institution_ids,
  pn_first,
  pn_first_initial,
  pn_last,
  work_source_ids,
  
  -- MATCH OUTCOME (s4_n8/s6_n8 retired, oxjob #691: 11%/26% judge-measured precision)
  CASE 
    WHEN om.orcid_author_id IS NOT NULL THEN 'MATCHED'
    WHEN (
      s1_n1=1 OR s1_n2=1 OR s1_n5=1 OR 
      s6_n1=1 OR s6_n2=1 OR s6_n5=1 OR s6_n6=1 OR
      s2_n1=1 OR s2_n2=1 OR s2_n5=1 OR s2_n6=1 OR s2_n8=1 OR
      s4_n1=1 OR s4_n2=1 OR s4_n5=1 OR s4_n6=1 OR
      s5_n1=1 OR s5_n2=1 OR s5_n5=1 OR s5_n6=1 OR s5_n8=1 OR
      s3_n1=1 OR s3_n2=1 OR s3_n5=1
    ) THEN 'MATCHED'
    WHEN total_candidates_in_block = 0 THEN 'NO_CANDIDATES'
    ELSE 'AMBIGUOUS'
  END AS match_outcome,

  -- NAME-BASED AUTHOR ID (existing name-pattern cascade)
  CASE 
    WHEN s1_n1 = 1 THEN match_s1_n1.id
    WHEN s1_n2 = 1 THEN match_s1_n2.id
    WHEN s1_n5 = 1 THEN match_s1_n5.id
    
    WHEN s6_n1 = 1 THEN match_s6_n1.id
    WHEN s6_n2 = 1 THEN match_s6_n2.id
    WHEN s6_n5 = 1 THEN match_s6_n5.id
    WHEN s6_n6 = 1 THEN match_s6_n6.id

    WHEN s2_n1 = 1 THEN match_s2_n1.id
    WHEN s2_n2 = 1 THEN match_s2_n2.id
    WHEN s2_n5 = 1 THEN match_s2_n5.id
    WHEN s2_n6 = 1 THEN match_s2_n6.id
    WHEN s2_n8 = 1 THEN match_s2_n8.id

    WHEN s4_n1 = 1 THEN match_s4_n1.id
    WHEN s4_n2 = 1 THEN match_s4_n2.id
    WHEN s4_n5 = 1 THEN match_s4_n5.id
    WHEN s4_n6 = 1 THEN match_s4_n6.id

    WHEN s5_n1 = 1 THEN match_s5_n1.id
    WHEN s5_n2 = 1 THEN match_s5_n2.id
    WHEN s5_n5 = 1 THEN match_s5_n5.id
    WHEN s5_n6 = 1 THEN match_s5_n6.id
    WHEN s5_n8 = 1 THEN match_s5_n8.id

    WHEN s3_n1 = 1 THEN match_s3_n1.id
    WHEN s3_n2 = 1 THEN match_s3_n2.id
    WHEN s3_n5 = 1 THEN match_s3_n5.id

    ELSE NULL
  END AS name_author_id,

  -- WHICH name tier fired (observational, oxjob #640): same WHEN order as
  -- name_author_id above, so this names the tier that produced that id.
  -- Consumed by AuthorshipDailyMetrics; no matching logic reads it.
  CASE 
    WHEN s1_n1 = 1 THEN 's1_n1'
    WHEN s1_n2 = 1 THEN 's1_n2'
    WHEN s1_n5 = 1 THEN 's1_n5'

    WHEN s6_n1 = 1 THEN 's6_n1'
    WHEN s6_n2 = 1 THEN 's6_n2'
    WHEN s6_n5 = 1 THEN 's6_n5'
    WHEN s6_n6 = 1 THEN 's6_n6'

    WHEN s2_n1 = 1 THEN 's2_n1'
    WHEN s2_n2 = 1 THEN 's2_n2'
    WHEN s2_n5 = 1 THEN 's2_n5'
    WHEN s2_n6 = 1 THEN 's2_n6'
    WHEN s2_n8 = 1 THEN 's2_n8'

    WHEN s4_n1 = 1 THEN 's4_n1'
    WHEN s4_n2 = 1 THEN 's4_n2'
    WHEN s4_n5 = 1 THEN 's4_n5'
    WHEN s4_n6 = 1 THEN 's4_n6'

    WHEN s5_n1 = 1 THEN 's5_n1'
    WHEN s5_n2 = 1 THEN 's5_n2'
    WHEN s5_n5 = 1 THEN 's5_n5'
    WHEN s5_n6 = 1 THEN 's5_n6'
    WHEN s5_n8 = 1 THEN 's5_n8'

    WHEN s3_n1 = 1 THEN 's3_n1'
    WHEN s3_n2 = 1 THEN 's3_n2'
    WHEN s3_n5 = 1 THEN 's3_n5'
    ELSE NULL
  END AS name_match_tier,

  -- ORCID tier (global, most-cited holder). orcid_match_count = GLOBAL number
  -- of profiles holding the ORCID (0 = no holder or no usable ORCID).
  COALESCE(om.orcid_match_count, 0) AS orcid_match_count,
  om.orcid_author_id

FROM aggregated_counts ac
LEFT JOIN orcid_matches om
  ON ac.work_id = om.work_id
  AND ac.author_sequence = om.author_sequence
)
SELECT
  *,
  -- FINAL AUTHOR ID: ORCID wins over the name cascade
  COALESCE(orcid_author_id, name_author_id) AS existing_author_id,
  CASE 
    WHEN orcid_author_id IS NOT NULL THEN 'orcid'
    WHEN name_author_id IS NOT NULL THEN 'name'
    ELSE NULL
  END AS match_method,
  -- QA: ORCID picked a different author than the name cascade would have
  (orcid_author_id IS NOT NULL AND name_author_id IS NOT NULL
   AND orcid_author_id <> name_author_id) AS orcid_name_conflict,
  -- QA: ORCID match with zero name corroboration (name cascade found nothing).
  -- This is the population to sample for wrong-ORCID publisher metadata.
  (orcid_author_id IS NOT NULL AND name_author_id IS NULL) AS orcid_blind_match
FROM final_decision;

### Step 3: Cluster Unmatched & Mint New IDs

In [ ]:
-- A. Get the current High Water Mark
DECLARE OR REPLACE VARIABLE max_id BIGINT;
SET VARIABLE max_id = (SELECT MAX(id) FROM openalex.authors.authors);

-- B. Cluster and Mint
CREATE OR REPLACE TABLE openalex.authors.author_matching_new_author_queue AS
WITH unmatched_with_hash AS (
    SELECT 
        pa.work_id,
        pa.author_sequence,
        pa.raw_author_name,
        b.authorship_struct.raw_orcid AS raw_orcid,
        
        xxhash64(
            -- 1. NAME PART: Normalized if available, else Raw
            CASE 
                WHEN pa.pn_first IS NOT NULL AND pa.pn_first != '' AND pa.pn_last IS NOT NULL
                THEN CONCAT(pa.pn_first, ' ', pa.pn_last)
                WHEN pa.pn_first_initial IS NOT NULL AND pa.pn_first_initial != '' AND pa.pn_last IS NOT NULL
                THEN CONCAT(pa.pn_first_initial, ' ', pa.pn_last)
                ELSE LOWER(TRIM(pa.raw_author_name))
            END,
            -- 2. SIGNAL PART: Institutions -> Sources
            CASE 
                WHEN SIZE(b.all_institution_ids) > 0
                THEN concat_ws('|', sort_array(b.all_institution_ids))
                ELSE concat_ws('|', sort_array(pa.work_source_ids))
            END
        ) AS cluster_hash

    FROM openalex.authors.pending_author_assignments pa
    --  Join Batch to get 'all_institution_ids'
    LEFT JOIN openalex.authors.author_matching_batch b
        ON pa.work_id = b.work_id AND pa.author_sequence = b.author_sequence
    LEFT JOIN openalex.works.work_authors existing
        ON pa.work_id = existing.work_id 
        AND pa.author_sequence = existing.author_sequence
    WHERE 
        -- Only unmatched records
        pa.match_outcome <> 'MATCHED'
        -- ensure we haven't already assigned an ID in a previous run
        AND existing.author_id IS NULL
        -- Safety: Ensure we actually have a name string to hash
        AND pa.raw_author_name IS NOT NULL 
        AND TRIM(pa.raw_author_name) <> ''
),
unique_clusters AS (
    SELECT 
        cluster_hash,
        MAX_BY(raw_author_name, length(raw_author_name)) as raw_display_name,
        MAX(raw_orcid) as orcid,
        monotonically_increasing_id() as batch_row_id
    FROM unmatched_with_hash
    GROUP BY cluster_hash
)
SELECT 
    uc.cluster_hash,
    CASE 
        WHEN SIZE(SPLIT(uc.raw_display_name, ',')) = 2 THEN 
            TRIM(SPLIT(uc.raw_display_name, ',')[1]) || ' ' || TRIM(SPLIT(uc.raw_display_name, ',')[0])
        ELSE 
            uc.raw_display_name 
    END AS display_name,
    uc.orcid,
    max_id + ROW_NUMBER() OVER (ORDER BY uc.batch_row_id) AS new_author_id
FROM unique_clusters uc;

In [0]:
-- logging: review match rates
SELECT 
    pa.match_outcome, 
    COUNT(*) as count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM openalex.authors.pending_author_assignments pa
GROUP BY pa.match_outcome

UNION ALL

-- 2. MINTING STATS
SELECT 
    'NEW_AUTHORS_TO_CREATE' as match_outcome,
    COUNT(*) as count,
    NULL as percentage
FROM openalex.authors.author_matching_new_author_queue;

### Step 4: Write New Authors to Profiles

In [ ]:
INSERT INTO openalex.authors.authors 
    (id, display_name, full_name, orcid, created_date, updated_date)
SELECT 
    new_author_id AS id,
    display_name,
    display_name AS full_name,
    orcid,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.authors.author_matching_new_author_queue;

### Step 5: Consolidate Decisions

In [ ]:
CREATE OR REPLACE TEMPORARY VIEW batch_author_decisions AS
SELECT 
    b.work_id,
    b.author_sequence,
    b.raw_author_name,
    
    COALESCE(pa.existing_author_id, q.new_author_id) AS final_author_id

FROM openalex.authors.author_matching_batch b

LEFT JOIN openalex.authors.pending_author_assignments pa
    ON b.work_id = pa.work_id 
    AND b.author_sequence = pa.author_sequence

LEFT JOIN openalex.authors.author_matching_new_author_queue q
    ON (pa.match_outcome IS NULL OR pa.match_outcome <> 'MATCHED')
    AND xxhash64(
            CASE 
               WHEN pa.pn_first IS NOT NULL AND pa.pn_first != '' AND pa.pn_last IS NOT NULL
               THEN CONCAT(pa.pn_first, ' ', pa.pn_last)
               WHEN pa.pn_first_initial IS NOT NULL AND pa.pn_first_initial != '' AND pa.pn_last IS NOT NULL
               THEN CONCAT(pa.pn_first_initial, ' ', pa.pn_last)
               ELSE LOWER(TRIM(b.raw_author_name))
            END,
            CASE 
                WHEN SIZE(b.all_institution_ids) > 0 
                THEN concat_ws('|', sort_array(b.all_institution_ids))
                ELSE concat_ws('|', sort_array(pa.work_source_ids))
            END
        ) = q.cluster_hash;

### Step 6: Update work_authors — author_id only

In [0]:
MERGE INTO openalex.works.work_authors AS target
USING batch_author_decisions AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence

WHEN MATCHED THEN
    UPDATE SET
        target.author_id = source.final_author_id,
        target.raw_author_name = source.raw_author_name,
        target.updated_at = current_timestamp()

WHEN NOT MATCHED THEN
    INSERT (work_id, author_sequence, author_id, raw_author_name, created_at, updated_at)
    VALUES (source.work_id, source.author_sequence, source.final_author_id,
            source.raw_author_name, current_timestamp(), current_timestamp())